## Import dataframe

In [1]:
import pandas as pd

In [2]:
stays_df = pd.read_feather('stays.feather')

In [3]:
stays_df.dtypes

Age                      int64
Admission               object
Discharge               object
ReAdmission             object
ReAdmissionDischarge    object
FirstPosCollected       object
Acquisition             object
dtype: object

In [4]:
stays_df.Admission = pd.to_datetime(stays_df.Admission, format='ISO8601')
stays_df.Discharge = pd.to_datetime(stays_df.Discharge, format='ISO8601')
stays_df.ReAdmission = pd.to_datetime(stays_df.ReAdmission, format='ISO8601')
stays_df.ReAdmissionDischarge = pd.to_datetime(stays_df.ReAdmissionDischarge, format='ISO8601')
stays_df.FirstPosCollected = pd.to_datetime(stays_df.FirstPosCollected, format='ISO8601')
stays_df.dtypes

Age                              int64
Admission               datetime64[ns]
Discharge               datetime64[ns]
ReAdmission             datetime64[ns]
ReAdmissionDischarge    datetime64[ns]
FirstPosCollected       datetime64[ns]
Acquisition                     object
dtype: object

In [5]:
stays_df

,Age,Admission,Discharge,ReAdmission,ReAdmissionDischarge,FirstPosCollected,Acquisition
0,88,2021-12-01 07:22:04.672,2021-12-01 12:08:15.104,2021-12-13 13:37:28.576,2021-12-13 16:45:20.768,2021-12-01 07:35:11.104,Community-Onset Community-Associated
3,48,2021-12-01 12:47:34.400,2021-12-07 19:29:21.408,2021-12-13 16:34:25.408,2021-12-19 09:26:05.056,2021-12-01 16:50:03.392,Community-Onset Community-Associated
4,39,2021-12-01 16:34:45.888,2021-12-04 18:59:57.056,NaT,NaT,2021-12-01 17:42:29.120,Community-Onset Community-Associated
5,72,2021-12-01 20:00:06.656,2021-12-23 14:39:05.216,NaT,NaT,2021-12-01 21:23:07.392,Community-Onset Community-Associated
6,73,2021-12-01 13:40:00.128,2021-12-02 20:04:05.248,NaT,NaT,2021-12-02 06:37:59.680,Community-Onset Community-Associated
...,...,...,...,...,...,...,...
5615,75,2024-05-31 11:29:12.192,2024-06-02 16:34:15.168,NaT,NaT,2024-05-31 17:20:54.784,Community-Onset Community-Associated
5616,56,2024-03-17 23:35:30.560,NaT,NaT,NaT,2024-06-01 02:37:58.144,Hospital-Onset Healthcare-Associated
5617,80,2024-06-03 11:10:32.960,2024-06-13 18:34:27.200,NaT,NaT,2024-06-03 11:38:56.896,Community-Onset Community-Associated
5618,88,2024-06-03 17:39:23.776,2024-06-12 11:59:26.720,NaT,NaT,2024-06-03 17:52:30.208,Community-Onset Community-Associated


## Filtering by Age

In [6]:
from itertools import pairwise
import re


def age_groups(age_breakpoints: str):
    # Validate str input
    age_breakpoints = '' if not age_breakpoints else age_breakpoints  # Handle None

    regex = r'^\d+(,\d+)*$'

    assert re.fullmatch(regex, age_breakpoints) is not None, \
        'Invalid input.  Expected a string of integers delimited by commas.'

    # Empty string case: single age group
    if age_breakpoints == '':
        return [{
            'lower': 0,
            'upper': None,
            'query': "Age >= 0"
        }]

    # Split str by comma and generate a dict for each age group
    age_breakpoints = age_breakpoints.split(',')
    age_breakpoints = [int(age) for age in age_breakpoints]

    assert all(x < y for x, y in pairwise(age_breakpoints)), \
        'Age breakpoints must be in strictly ascending order with no duplicates.'

    pairs = list(pairwise([0] + age_breakpoints + [None]))
    return [
        {
            'lower': pair[0],
            'upper': pair[1],
            'query': f"Age >= {pair[0]}{f' and Age < {pair[1]}' if pair[1] is not None else ''}"
        }
        for pair in pairs
    ]

In [7]:
def test_age_groups(s: str):
    try:
        return age_groups(s)
    except AssertionError as e:
        return str(e)

In [8]:
display(test_age_groups('16,65'))
display(test_age_groups('16, 65'))
display(test_age_groups('16.65'))

[{'lower': 0, 'upper': 16, 'query': 'Age >= 0 and Age < 16'},
 {'lower': 16, 'upper': 65, 'query': 'Age >= 16 and Age < 65'},
 {'lower': 65, 'upper': None, 'query': 'Age >= 65'}]

'Invalid input.  Expected a string of integers delimited by commas.'

'Invalid input.  Expected a string of integers delimited by commas.'

In [9]:
display(test_age_groups('65,16'))
display(test_age_groups('16,16'))

'Age breakpoints must be in strictly ascending order with no duplicates.'

'Age breakpoints must be in strictly ascending order with no duplicates.'

In [10]:
display(test_age_groups('65'))
display(test_age_groups(''))

[{'lower': 0, 'upper': 65, 'query': 'Age >= 0 and Age < 65'},
 {'lower': 65, 'upper': None, 'query': 'Age >= 65'}]

'Invalid input.  Expected a string of integers delimited by commas.'

## 

In [11]:
display(test_age_groups('0,,16'))

'Invalid input.  Expected a string of integers delimited by commas.'

## LoS fitting

In [12]:
g = age_groups('16,65')

In [13]:
query = g[2]['query']
query

'Age >= 65'

In [14]:
df = stays_df.query(query)
df

,Age,Admission,Discharge,ReAdmission,ReAdmissionDischarge,FirstPosCollected,Acquisition
0,88,2021-12-01 07:22:04.672,2021-12-01 12:08:15.104,2021-12-13 13:37:28.576,2021-12-13 16:45:20.768,2021-12-01 07:35:11.104,Community-Onset Community-Associated
5,72,2021-12-01 20:00:06.656,2021-12-23 14:39:05.216,NaT,NaT,2021-12-01 21:23:07.392,Community-Onset Community-Associated
6,73,2021-12-01 13:40:00.128,2021-12-02 20:04:05.248,NaT,NaT,2021-12-02 06:37:59.680,Community-Onset Community-Associated
9,94,2021-12-02 19:59:43.104,2021-12-14 15:00:05.760,NaT,NaT,2021-12-02 22:04:14.208,Community-Onset Community-Associated
10,66,2021-12-03 10:20:25.472,2021-12-18 19:59:59.488,NaT,NaT,2021-12-03 11:21:35.488,Community-Onset Community-Associated
...,...,...,...,...,...,...,...
5613,78,2024-05-30 22:16:13.056,2024-06-03 16:44:46.976,NaT,NaT,2024-05-30 22:33:41.632,Community-Onset Community-Associated
5614,89,2024-05-09 19:51:32.608,NaT,NaT,NaT,2024-05-31 13:55:34.016,Hospital-Onset Healthcare-Associated
5615,75,2024-05-31 11:29:12.192,2024-06-02 16:34:15.168,NaT,NaT,2024-05-31 17:20:54.784,Community-Onset Community-Associated
5617,80,2024-06-03 11:10:32.960,2024-06-13 18:34:27.200,NaT,NaT,2024-06-03 11:38:56.896,Community-Onset Community-Associated
